# generate_questions — research notebook

The main LLM call: one question per blueprint slot, grounded strictly in the clean text. Also covers the **schema gate** that sits right after it in the graph.

Two functions, two jobs:

- `generate_questions(...)` — one batched LLM call for the given slots (repair passes it only the failed slots), never raises
- `validate_questions(...)` — the deterministic gate: index in range, options sane, no repeats of recent questions; pure Python, free

Needs `OPENAI_API_KEY` in `research/.env` for the live cells.

In [ ]:
import os, hashlib
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(".env")

BloomLevel = Literal["remember", "understand", "apply", "analyse", "evaluate", "create"]

class Slot(TypedDict):
    slot_id: int
    concept: str
    bloom_level: BloomLevel
    is_variant: bool

class QuizQuestion(TypedDict):
    slot_id: int
    question: str
    options: list[str]       # exactly 4
    correct_answer: int      # index into options
    explanation: str

class QuestionBatch(TypedDict):
    questions: list[QuizQuestion]

class GenerateResult(TypedDict):
    ok: bool
    questions: list[QuizQuestion]
    error: str

def qhash(q: str) -> str:
    return hashlib.sha256(q.lower().strip().encode()).hexdigest()[:12]

llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gpt-5-mini"))
generator = llm.with_structured_output(QuestionBatch)

## Generation

One batched call for all requested slots: cheaper than per-slot calls, and `repair_questions` reuses it by passing only the failed slots plus the critique feedback.

In [ ]:
SYSTEM = """You write multiple-choice quiz questions grounded ONLY in the provided article.
For each requested slot, write one question testing the given concept at the given Bloom level:
remember=recall a fact, understand=explain it, apply=use it in a new situation,
analyse=relate or contrast it, evaluate=judge a claim about it, create=extend it.
Rules:
- The answer must be derivable from the article alone.
- Exactly 4 options, one correct, three plausible but clearly wrong distractors.
- Distractors must not be trick wordings unless allowed.
- Write a one-sentence explanation citing the article's reasoning.
- Questions must differ from each other and from the recent questions listed."""

def generate_questions(clean_text: str, slots: list[Slot],
                       recent_hashes: list[str] | None = None,
                       feedback: str = "") -> GenerateResult:
    """Never raises: LLM failures return ok=False."""
    try:
        slot_lines = "\n".join(
            f'- slot_id {s["slot_id"]}: concept "{s["concept"]}", bloom level {s["bloom_level"]}'
            for s in slots)
        user = f"Article:\n{clean_text}\n\nSlots:\n{slot_lines}"
        if feedback:
            user += f"\n\nA reviewer rejected earlier attempts. Fix these issues:\n{feedback}"
        out: QuestionBatch = generator.invoke([("system", SYSTEM), ("user", user)])
        return GenerateResult(ok=True, questions=out["questions"], error="")
    except Exception as e:                       # noqa: BLE001 — resilience boundary
        return GenerateResult(ok=False, questions=[], error=f"Generation failed: {e}")

## The schema gate

Deterministic and free. Anything the gate rejects goes back to `generate_questions` (max 2 retries) with the reasons as feedback.

In [ ]:
def validate_questions(questions: list[QuizQuestion], slots: list[Slot],
                       recent_hashes: list[str] | None = None) -> tuple[list[QuizQuestion], list[str]]:
    """Returns (valid_questions, rejection_reasons). Pure function, never raises."""
    recent = set(recent_hashes or [])
    want = {s["slot_id"] for s in slots}
    valid, reasons, seen_hashes = [], [], set()

    for q in questions or []:
        sid = q.get("slot_id")
        h = qhash(q.get("question", ""))
        if sid not in want:
            reasons.append(f"slot {sid}: not a requested slot"); continue
        if len(q.get("options", [])) != 4:
            reasons.append(f"slot {sid}: needs exactly 4 options"); continue
        if len(set(q["options"])) != 4:
            reasons.append(f"slot {sid}: duplicate options"); continue
        if not 0 <= q.get("correct_answer", -1) < 4:
            reasons.append(f"slot {sid}: correct_answer out of range"); continue
        if h in recent or h in seen_hashes:
            reasons.append(f"slot {sid}: repeats a recent question"); continue
        seen_hashes.add(h)
        valid.append(q)

    for sid in sorted(want - {q["slot_id"] for q in valid}):
        if not any(f"slot {sid}" in r for r in reasons):
            reasons.append(f"slot {sid}: missing")
    return valid, reasons

## Gate tests (offline, mocked model output)

In [ ]:
slots: list[Slot] = [
    {"slot_id": 0, "concept": "multi-head attention", "bloom_level": "analyse", "is_variant": False},
    {"slot_id": 1, "concept": "positional encoding", "bloom_level": "analyse", "is_variant": False},
    {"slot_id": 2, "concept": "self-attention", "bloom_level": "evaluate", "is_variant": False},
]
mock = [
    {"slot_id": 0, "question": "How do the heads in multi-head attention differ?",
     "options": ["a", "b", "c", "d"], "correct_answer": 1, "explanation": "..."},
    {"slot_id": 1, "question": "Why does the model need positional encodings?",
     "options": ["a", "b", "c", "d"], "correct_answer": 7, "explanation": "..."},   # bad index
    {"slot_id": 2, "question": "Evaluate: attention removes recurrence entirely.",
     "options": ["a", "a", "c", "d"], "correct_answer": 0, "explanation": "..."},   # dup options
]
valid, reasons = validate_questions(mock, slots, recent_hashes=[])
assert [q["slot_id"] for q in valid] == [0]
assert any("out of range" in r for r in reasons) and any("duplicate" in r for r in reasons)
print("kept:", len(valid), "| rejected:", reasons)

# repeat blocking
valid2, r2 = validate_questions(mock[:1], slots[:1], recent_hashes=[qhash(mock[0]["question"])])
assert not valid2 and any("repeats" in r for r in r2)
print("repeat blocked:", r2)
print("gate tests passed")

## Live run (needs API key)

Chains the whole research stack so far: article text → slots → questions.

In [ ]:
ARTICLE = (
    "The transformer replaces recurrence with self-attention, letting every token attend to "
    "every other token in one step. Multi-head attention runs several attention functions in "
    "parallel so different heads can focus on different relationships, such as syntax versus "
    "meaning. Because the model sees tokens as an unordered set, positional encodings are "
    "added to the embeddings so word order still matters."
)
res = generate_questions(ARTICLE, slots, recent_hashes=[])
if res["ok"]:
    valid, reasons = validate_questions(res["questions"], slots)
    for q in valid:
        print(f'[{q["slot_id"]}] {q["question"]}')
        for i, o in enumerate(q["options"]):
            print(("  -> " if i == q["correct_answer"] else "     ") + o)
        print("     ", q["explanation"], "\n")
    print("gate rejections:", reasons or "none")
else:
    print("graceful failure:", res["error"])

## Notes

- **Batch call, slot-addressable:** `repair_questions` calls the same function with only the failed slots and the critique feedback — no separate repair prompt to maintain.
- The gate is a separate node in the graph so the retry edge is visible and checkpointed; retry max 2, then the run proceeds with the valid subset (fewer questions beats a dead run).
- Trick questions: allowed only when `profile.allow_trick_questions`; wire the flag into SYSTEM when the profile plumbing lands.
- Use the stronger model here if quality demands it; analysis and critique can stay on the small one.
- The grounding judgment (is the answer really supported by the article?) is NOT this node's job — that's `critique_questions`, next.